# K 值研究

看不同状态数下 A 股市场被切成了什么样。逻辑在 `experiment/k_study.py`，这里只调用、看图。

**图直接出在下面**，不落盘。

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import display

from experiment import k_study

feats, close = k_study.load_features("ashare")
regular, ext = k_study.split_extreme(feats, "ashare")
print(f"特征 {feats.shape}  {feats.index[0].date()} ~ {feats.index[-1].date()}")
print(f"常规日 {len(regular)}  极端日 {int((ext != '').sum())}")
feats.tail(3)

## 逐个 K 拟合

- `separation`：状态间 Bhattacharyya 最小距离，越大越分得开
- `min_share` / `min_duration` / `min_stay`：状态是否"有用"——占比太小的、平均只持续一两天的、留不住自己的，都是退化状态


In [ ]:
ks = (2, 3, 4, 5, 6)
res = {}
rows = []
for k in ks:
    r = k_study.fit_k(regular, k)
    res[k] = r
    prof = k_study.profile(r, regular)
    rows.append({"k": k, "separation": round(r["sep"]["min"], 3),
                 "bic": round(r["bic"]),
                 "min_share": round(prof["share"].min(), 3),
                 "min_duration": round(prof["avg_duration"].min(), 1),
                 "min_stay": round(prof["stay_prob"].min(), 3)})

summary = pd.DataFrame(rows)
summary

## 每个 K 的状态画像

In [ ]:
for k in ks:
    print(f"===== K={k}  separation={res[k]['sep']['min']:.2f} =====")
    display(k_study.profile(res[k], regular).round(3))
    print()

## 市场分割图

颜色块 = 同一个状态连续的区间。看它是否对应真实行情段落（2018 熊市、2021 结构市、2024 反转）。

In [ ]:
import matplotlib.pyplot as plt

for k in ks:
    fig = k_study.segment_figure(
        res[k], close.reindex(regular.index),
        f"ashare  K={k}  separation={res[k]['sep']['min']:.2f}  BIC={res[k]['bic']:.0f}")
    display(fig)
    plt.close(fig)

## 怎么读

- 碎块多 ⇒ 状态在追噪声
- 某状态天天只出现一两根 ⇒ 它是"异常点收纳盒"，没有市场含义
- 同一段行情里颜色反复跳 ⇒ 特征没抓住这段行情，不是 K 的问题
- 所有 K 都只按波动率分层 ⇒ 该换特征，不是调 K
